# Spatio-Temporal Declustering of the New Zealand Earthquake Catalog
## Using a Two-Stage Self-Organizing Map and Density-Based Clustering Framework

---

**Author:** [Md Ashraf]  
**Institution:** [IIT ISM Dhanbad]
**Programme:** MSc (Tech) Applied Geophysics
**Supervisors:** Dr. Niptika Jana
**Date:** 25 March 2026

---

## 1. Introduction and Motivation

Earthquake catalogs are the primary data source for understanding seismicity patterns, estimating fault activity, and performing probabilistic seismic hazard analysis (PSHA). However, raw earthquake catalogs contain two fundamentally different types of events that must not be mixed in statistical analyses:

**Background events (BGs)** are independent earthquakes driven by long-term tectonic loading. They occur randomly in time, following an approximately stationary Poisson process. These are the events relevant to hazard estimation because they reflect the underlying seismogenic behaviour of faults.

**Aftershock sequences (AFs)** are short-term, correlated events triggered by a preceding mainshock through stress redistribution in the crust. They decay in frequency over time according to Omori's law and can dominate a catalog for days to months following a large earthquake. Including them in hazard analyses inflates the apparent seismicity rate and violates the Poisson assumption.

The process of separating these two populations is called **seismicity declustering**.

### Why New Zealand?

New Zealand sits at the boundary of the Australian and Pacific tectonic plates, making it one of the most seismically active regions in the world. The catalog is characterised by:
- Distributed shallow seismicity across the North Island volcanic arc
- Deep subduction-related events in the Hikurangi margin
- Strike-slip faulting in the South Island (Alpine Fault system)
- Complex multi-zone fault geometries that challenge simple window-based declustering methods

This spatial heterogeneity makes New Zealand an ideal test case for a method that explicitly models spatial structure before performing temporal clustering.

### My Approach

I implement a **two-stage machine learning pipeline** for declustering:

1. **Stage 1 — Self-Organizing Map (SOM):** An unsupervised neural network is trained on event locations (latitude, longitude, depth) to partition the catalog into spatially coherent seismotectonic zones. This avoids the arbitrary rectangular windows used by classical methods.

2. **Stage 2 — Temporal DBSCAN (T-DBSCAN):** Within each spatial zone, a density-based clustering algorithm identifies groups of events that are anomalously close in time — these are classified as aftershock sequences. Events that cannot be assigned to any dense temporal cluster are retained as background seismicity.

The method is validated using the **Coefficient of Variation (COV)** of inter-event times and the **m-Morisita spatial clustering index**, and cross-validated against ETAS (Epidemic Type Aftershock Sequence) model outputs already computed for the catalog.

---


---
## Cell 1: Environment Setup

In [1]:
# =============================================================================
# CELL 1 — Environment Setup
#
# All packages used in this notebook are standard open-source scientific
# Python libraries. MiniSom is a lightweight, well-documented SOM
# implementation (Vettigli, 2018) that allows full control over training
# parameters, making it suitable for research rather than black-box use.
#
# Install if needed:
#   pip install minisom scikit-learn pandas numpy matplotlib seaborn scipy
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import cdist
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from minisom import MiniSom
import warnings
import os

warnings.filterwarnings('ignore')

# Reproducibility — fix random seed for all stochastic operations
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Create output directory for figures
os.makedirs('figures', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Global plot settings
plt.rcParams.update({
    'figure.dpi'    : 130,
    'font.size'     : 11,
    'font.family'   : 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

print("Environment ready.")
print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")
print(f"Random seed fixed at: {RANDOM_SEED}")

Environment ready.
NumPy  : 1.26.4
Pandas : 2.1.4
Random seed fixed at: 42


---
## Cell 2: Data Loading and Inspection

### Catalog Structure

The New Zealand earthquake catalog used here was processed using an ETAS (Epidemic Type Aftershock Sequence) model prior to this analysis. Each row represents one earthquake event and contains:

| Column | Description |
|--------|-------------|
| `event` | Event index |
| `time` | Occurrence time in decimal days from catalog origin |
| `latitude` | Epicentre latitude (decimal degrees, negative = South) |
| `longitude` | Epicentre longitude (decimal degrees) |
| `magnitude` | Local magnitude (Richter scale) |
| `i+` | Number of preceding events in the ETAS neighbourhood |
| `N+` | ETAS background rate estimate |
| `T+` | ETAS temporal influence measure |
| `R+` | ETAS branching ratio — probability this event is an aftershock |
| `dm+` | Magnitude difference from triggering parent event |
| `n_child` | Expected number of direct offspring events (ETAS) |
| `n_parent` | Expected number of ancestors (ETAS) |

The ETAS columns will **not** be used as inputs to the SOM–DBSCAN algorithm — they serve as an independent cross-validation reference in Cell 11.

In [ ]:
# =============================================================================
# CELL 2 — Load and Inspect Catalog
#
# For production use, replace the inline dictionary with:
#   df_raw = pd.read_csv('path/to/NZ_catalog.csv')
#
# Required columns: time, latitude, longitude, magnitude
# Optional but recommended: depth
# =============================================================================

# ── Inline sample catalog (replace with full NZ catalog for thesis) ──────────
catalog_dict = {
    'event'    : [3, 4, 5, 7, 8, 9, 11, 12, 13, 15, 20, 21, 25, 26, 27],
    'time'     : [0.4750623843, 0.5533383102, 1.4723356481, 2.7669097222,
                  2.8091324074, 3.1366680556, 3.5616989583, 4.2160844907,
                  4.5041237269, 4.6458788194, 5.9792454861, 6.0694148148,
                  7.8190607639, 8.0006599537, 8.1780275463],
    'latitude' : [-37.63, -40.29, -38.48, -36.66, -38.71, -41.625, -44.329,
                  -46.69, -38.7, -37.52, -38.69, -40.945, -46.78, -37.73, -36.55],
    'longitude': [176.38, 173.42999, 176.75, 177.45, 175.94, 174.23, 169.57001,
                  165.62, 175.61, 177.06, 175.83, 174.87, 165.49001, 179.02, 177.35001],
    'magnitude': [4.15, 4.5, 3.375, 4.818, 4.011, 3.2, 3.301, 3.878, 3.55,
                  3.962, 4.743, 3.15, 4.214, 4.135, 3.918],
    'i+'       : [0, 1, 1, 1, 4, 4, 4, 4, 5, 4, 5, 11, 8, 4, 4],
    'N+'       : [0.00130437198, 3.78e-5, 4.70e-5, 2.19e-4, 2.05e-6, 6.93e-5,
                  4.01e-4, 1.27e-3, 2.13e-5, 1.87e-5, 7.04e-6, 4.93e-6,
                  1.91e-5, 1.59e-4, 2.79e-6],
    'T+'       : [0.00845126, 5.71e-4, 7.27e-3, 1.67e-2, 1.39e-4, 1.22e-3,
                  2.62e-3, 4.78e-3, 1.46e-2, 6.20e-3, 2.73e-2, 3.25e-4,
                  3.63e-2, 1.73e-2, 1.79e-2],
    'R+'       : [0.15434058, 0.06625140, 0.00645812, 0.01312708, 0.01472751,
                  0.05677788, 0.15300072, 0.26527358, 0.00145974, 0.00301235,
                  2.58e-4, 0.01514466, 5.26e-4, 9.22e-3, 1.56e-4],
    'dm+'      : [-0.95, -0.35, 0.775, -0.668, 0.807, 1.618, 1.517, 0.94,
                  0.461, 0.856, -0.732, 1.593, -0.336, 0.683, 0.9],
    'n_child'  : [0.5174, 0.2587, 0.0, 1.5523, 0.2587, 0.1294, 0.0, 0.2587,
                  0.0, 0.0, 1.2936, 0.0, 0.3881, 0.0, 0.1294],
    'n_parent' : [0.1294, 0.5174, 0.5174, 0.5174, 1.5523, 1.5523, 1.5523,
                  1.5523, 0.2587, 1.5523, 0.2587, 1.2936, 0.2587, 1.5523, 1.5523]
}

df_raw = pd.DataFrame(catalog_dict)

# ── Add depth column ─────────────────────────────────────────────────────────
# Depth is not in the sample dataset. I assign a representative shallow
# crustal depth here. For the full thesis catalog, replace with actual values.
if 'depth' not in df_raw.columns:
    df_raw['depth'] = 15.0
    print("Note: depth column not found — set to 15 km placeholder.")
    print("Replace with actual depth values from the full catalog.")

# ── Sort chronologically ──────────────────────────────────────────────────────
df_raw = df_raw.sort_values('time').reset_index(drop=True)

# ── Summary statistics ────────────────────────────────────────────────────────
print(f"\nCatalog loaded successfully.")
print(f"Total events      : {len(df_raw)}")
print(f"Time span         : {df_raw['time'].min():.3f} – {df_raw['time'].max():.3f} days")
print(f"Latitude range    : {df_raw['latitude'].min():.2f}° – {df_raw['latitude'].max():.2f}°")
print(f"Longitude range   : {df_raw['longitude'].min():.2f}° – {df_raw['longitude'].max():.2f}°")
print(f"Magnitude range   : {df_raw['magnitude'].min():.2f} – {df_raw['magnitude'].max():.2f}")
print()
df_raw[['event','time','latitude','longitude','magnitude','depth']].head(8)

---
## Cell 3: Magnitude of Completeness (Mc)

### Why Mc matters

Seismic networks have a detection threshold. Small earthquakes are frequently missed — either because their signal amplitude is below the noise floor on individual seismometers, or because an insufficient number of stations recorded the phase arrival to allow a reliable location. The **Magnitude of Completeness (Mc)** is the lowest magnitude above which the network reliably detects every earthquake in the region.

Including events below Mc introduces **systematic incompleteness** that creates artificial clustering signals: gaps between detected events appear shorter than they truly are, which can cause the temporal clustering algorithm to incorrectly merge distinct background events into aftershock sequences.

### Gutenberg–Richter Law

The frequency–magnitude distribution of earthquakes follows the empirical Gutenberg–Richter (GR) relation:

$$\log_{10} N(m) = a - b \cdot m$$

where $N(m)$ is the cumulative number of events with magnitude $\geq m$, $b \approx 1$ is the slope (b-value), and $a$ is a constant related to regional productivity. This relationship holds as a power law across many orders of magnitude, but **deviates at low magnitudes** due to network incompleteness. The point of deviation defines Mc.

### My Estimation Method: Maximum Curvature

I use the **Maximum Curvature Method** to determine Mc. The non-cumulative frequency–magnitude distribution (FMD) — the histogram of event counts per magnitude bin — rises steeply from high magnitudes, reaches a peak, and then drops at small magnitudes as the network misses events. The peak of this distribution is Mc:

$$M_c = \underset{m}{\arg\max} \left[ \frac{d\,N_{\text{discrete}}}{dm} \right]$$

This is computationally simple and requires fewer events than maximum likelihood alternatives, making it suitable for regional sub-catalogs. All events with $m < M_c$ are discarded before the declustering pipeline begins.

In [ ]:
# =============================================================================
# CELL 3 — Magnitude of Completeness Estimation
# =============================================================================

def estimate_magnitude_completeness(magnitudes, bin_width=0.1):
    """
    Estimate Mc using the Maximum Curvature Method.

    The non-cumulative FMD is binned at 'bin_width' intervals. Mc is the
    centre of the bin with the highest event count. The b-value is then
    estimated by least-squares linear regression of log10(N) on magnitude
    for all bins at or above Mc.

    Parameters
    ----------
    magnitudes : array-like  — raw magnitude values
    bin_width  : float       — histogram bin width (default 0.1, standard)

    Returns
    -------
    Mc         : float  — magnitude of completeness
    b_value    : float  — Gutenberg-Richter b-value
    a_value    : float  — Gutenberg-Richter a-value
    bin_centers: array  — bin midpoints for plotting
    freq       : array  — non-cumulative counts per bin
    cum_freq   : array  — cumulative counts (N >= m)
    """
    mags = np.array(magnitudes, dtype=float)

    # Build histogram bins covering the full magnitude range
    m_lo   = np.floor(mags.min() / bin_width) * bin_width
    m_hi   = np.ceil(mags.max()  / bin_width) * bin_width
    edges  = np.arange(m_lo, m_hi + bin_width, bin_width)
    freq, _ = np.histogram(mags, bins=edges)
    centers = edges[:-1] + bin_width / 2.0

    # Mc = centre of the peak bin in the non-cumulative FMD
    Mc = centers[np.argmax(freq)]

    # Cumulative FMD: N(m) = number of events with magnitude >= m
    cum_freq = np.array([np.sum(mags >= m) for m in centers])

    # b-value via least-squares fit on the complete portion (m >= Mc)
    fit_mask = (centers >= Mc) & (cum_freq > 0)
    if fit_mask.sum() > 1:
        log_N = np.log10(cum_freq[fit_mask].astype(float) + 1e-10)
        slope, intercept, r, p, se = stats.linregress(centers[fit_mask], log_N)
        b_value = -slope
        a_value = intercept + b_value * Mc
    else:
        b_value, a_value = np.nan, np.nan

    return Mc, b_value, a_value, centers, freq, cum_freq


# ── Run estimation ────────────────────────────────────────────────────────────
Mc, b_value, a_value, bin_centers, freq_counts, cum_counts = \
    estimate_magnitude_completeness(df_raw['magnitude'])

print(f"Magnitude of Completeness  Mc = {Mc:.2f}")
print(f"Gutenberg-Richter b-value   b = {b_value:.3f}")
print(f"Gutenberg-Richter a-value   a = {a_value:.3f}")

# ── Plot FMD ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4.5))

# Non-cumulative bars
ax.bar(bin_centers, freq_counts, width=0.085,
       color='steelblue', alpha=0.5, edgecolor='steelblue', label='Discrete count')

# Cumulative line
ax_r = ax.twinx()
ax_r.semilogy(bin_centers, cum_counts + 0.1, 'ko-', ms=4, lw=1.5,
              label='Cumulative N ≥ M')

# G-R fit line
if not np.isnan(b_value):
    m_fit = bin_centers[bin_centers >= Mc]
    n_fit = 10 ** (a_value - b_value * m_fit)
    ax_r.semilogy(m_fit, n_fit, 'r--', lw=2,
                  label=f'G-R fit  b={b_value:.2f}')

# Mc line
ax.axvline(Mc, color='crimson', lw=2, ls='-',
           label=f'Mc = {Mc:.2f}')

ax.set_xlabel('Magnitude')
ax.set_ylabel('Non-cumulative count (bars)', color='steelblue')
ax_r.set_ylabel('Cumulative count  log scale (line)', color='black')
ax.set_title('Frequency–Magnitude Distribution and Magnitude of Completeness')

# Combined legend
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax_r.get_legend_handles_labels()
ax.legend(lines1 + lines2, labs1 + labs2, fontsize=9, loc='upper right')
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('figures/01_fmd.png', dpi=130, bbox_inches='tight')
plt.show()

# ── Apply filter ──────────────────────────────────────────────────────────────
# Manually override Mc here if the automatic estimate looks wrong
# e.g. Mc = 3.5
df = df_raw[df_raw['magnitude'] >= Mc].copy().reset_index(drop=True)

print(f"\nEvents before Mc filter : {len(df_raw)}")
print(f"Events after  Mc filter : {len(df)}   (Mc = {Mc:.2f})")

if len(df) < 50:
    print("\nWarning: fewer than 50 events after filtering.")
    print("SOM and DBSCAN results are illustrative for this sample.")
    print("Load the full NZ catalog for thesis-quality results.")

---
## Cell 4: Spatial Distance Metric

### Hypocenter Separation Distance

The SOM must measure how similar two earthquake locations are. I use a **3-dimensional hypocenter distance** that combines horizontal surface separation and focal depth difference. This is more physically meaningful than treating latitude, longitude, and depth as if they were points in a flat Euclidean space — particularly important for New Zealand where shallow crustal events (5–20 km) and deep subduction events (>100 km) should not be clustered together even if their epicentres coincide.

The horizontal surface distance between events $i$ and $j$ is the **great-circle distance** on a spherical Earth of radius $R_E = 6371$ km:

$$R_{ij} = R_E \cdot \arccos\!\left[\sin\phi_i \sin\phi_j + \cos\phi_i \cos\phi_j \cos|\theta_j - \theta_i|\right]$$

where $\phi$ is latitude and $\theta$ is longitude in radians. The full 3D hypocenter separation is then:

$$\eta_{ij} = \sqrt{R_{ij}^2 + (d_j - d_i)^2}$$

where $d_i$ and $d_j$ are focal depths in kilometres. This metric is used inside the SOM training loop to compute distances between input events and neuron weight vectors.

> **Implementation note:** The MiniSom library by default uses Euclidean distance in the normalised input space. Since latitude and longitude differences in normalised space do not faithfully represent great-circle arcs, I use MinMaxScaling as a practical approximation for the SOM phase (where only relative ordering matters), and apply the exact hypocenter formula for the IET–IED scatter plots in Cell 9.

In [ ]:
# =============================================================================
# CELL 4 — Spatial Distance Functions
# =============================================================================

EARTH_RADIUS_KM = 6371.0


def great_circle_km(lat1_deg, lon1_deg, lat2_deg, lon2_deg):
    """
    Compute the great-circle surface distance between two geographic
    points in kilometres.

    Uses the spherical law of cosines. For seismological distances
    (typically 1–2000 km) this is numerically stable and sufficiently
    accurate compared to the Vincenty ellipsoidal formula.

    Parameters
    ----------
    lat1_deg, lon1_deg : float — coordinates of point 1 (decimal degrees)
    lat2_deg, lon2_deg : float — coordinates of point 2 (decimal degrees)

    Returns
    -------
    distance : float — surface distance in km
    """
    lat1 = np.radians(lat1_deg)
    lat2 = np.radians(lat2_deg)
    dlon = np.radians(np.abs(lon2_deg - lon1_deg))

    # Clamp argument of arccos to [-1, 1] to avoid NaN from floating-point
    dot = np.clip(
        np.sin(lat1) * np.sin(lat2) + np.cos(lat1) * np.cos(lat2) * np.cos(dlon),
        -1.0, 1.0
    )
    return EARTH_RADIUS_KM * np.arccos(dot)


def hypocenter_distance_km(lat1, lon1, depth1_km, lat2, lon2, depth2_km):
    """
    Compute the full 3D hypocenter-to-hypocenter distance in km.
    Combines horizontal great-circle distance with vertical depth difference.

    Parameters
    ----------
    lat1, lon1, depth1_km : location of event 1
    lat2, lon2, depth2_km : location of event 2

    Returns
    -------
    eta : float — 3D separation distance in km
    """
    R  = great_circle_km(lat1, lon1, lat2, lon2)
    dz = depth2_km - depth1_km
    return np.sqrt(R**2 + dz**2)


# ── Vectorised pairwise distance matrix ──────────────────────────────────────
def pairwise_hypocenter_matrix(df_in):
    """
    Build the full symmetric N×N pairwise hypocenter distance matrix.
    Used for diagnostic purposes and IET–IED analysis.

    For large catalogs (N > 5000) this is O(N²) and may be slow.
    Consider computing only for consecutive pairs in that case.
    """
    n = len(df_in)
    D = np.zeros((n, n))
    lats   = df_in['latitude'].values
    lons   = df_in['longitude'].values
    depths = df_in['depth'].values
    for i in range(n):
        for j in range(i + 1, n):
            d = hypocenter_distance_km(lats[i], lons[i], depths[i],
                                       lats[j], lons[j], depths[j])
            D[i, j] = d
            D[j, i] = d
    return D


# ── Sanity check ──────────────────────────────────────────────────────────────
d_test = great_circle_km(-37.63, 176.38, -40.29, 173.43)
print(f"Distance check (events 3→4): {d_test:.1f} km  (expect ~350 km)")

eta_test = hypocenter_distance_km(-37.63, 176.38, 15.0, -40.29, 173.43, 15.0)
print(f"3D hypocenter distance    : {eta_test:.1f} km")
print("Distance functions ready.")

---
## Cell 5: Stage 1 — Self-Organizing Map for Seismic Zonation

### What is a Self-Organizing Map?

A Self-Organizing Map (SOM) is an **unsupervised artificial neural network** that learns a topology-preserving mapping from a high-dimensional input space onto a two-dimensional grid of neurons. Unlike k-means or hierarchical clustering, SOM preserves the **neighbourhood relationships** of the input data — events that are close in 3D space are mapped to neurons that are close on the 2D grid.

The SOM was originally proposed by Kohonen (1982) and has since been applied in seismology for tectonic zoning, seismogenic source identification, and fault geometry analysis.

### Algorithm: SOM Training

I train the SOM on the 3-feature vector $x_i = [\phi_i, \theta_i, d_i]$ (latitude, longitude, depth) for each event $e_i$. The grid consists of $m = g_x \times g_y$ neurons, each carrying a weight vector $w_l \in \mathbb{R}^3$.

**Training proceeds iteratively:**

**Step 1 — Initialisation:** Neuron weights $w_l$ are initialised by sampling randomly from the input data distribution.

**Step 2 — Random input selection:** An event $x_i$ is drawn at random from the catalog.

**Step 3 — Best Matching Unit (BMU):** The winning neuron $c$ is the one whose weight vector is closest to $x_i$ in Euclidean distance:

$$c = \underset{l}{\arg\min} \; \|x_i - w_l\|_2 \quad \forall \; l = 1, 2, \ldots, m$$

**Step 4 — Weight update:** The BMU and its topological neighbours are pulled toward $x_i$:

$$w_l(t+1) = w_l(t) + \alpha(t) \cdot h_{c,l}(t) \cdot \left[x_i(t) - w_l(t)\right]$$

where $\alpha(t)$ is the **learning rate** that decays monotonically from $\alpha_0$ to 0 over $T$ iterations, and $h_{c,l}(t)$ is the **Gaussian neighbourhood function**:

$$h_{c,l}(t) = \exp\!\left(-\frac{\|r_c - r_l\|^2}{2\sigma^2(t)}\right)$$

Here $r_c$ and $r_l$ are the grid positions of neurons $c$ and $l$, and $\sigma(t)$ is the neighbourhood radius that also decays from $\sigma_0$ toward zero. At early iterations, many neurons are updated (global organisation); at later iterations, only the BMU and its immediate neighbours are updated (local fine-tuning).

**Step 5 — Repeat** Steps 2–4 for all events, then for $T$ iterations until convergence.

### Convergence Criterion: Quantization Error

Convergence is measured by the **Quantization Error (QE)**, defined as the mean Euclidean distance between each input event and its BMU:

$$QE = \frac{1}{N} \sum_{i=1}^{N} \|x_i - w_{c(i)}\|_2$$

A small QE means the SOM weight vectors accurately represent the input distribution. I evaluate QE across multiple grid sizes and select the grid that achieves the minimum QE — this grid provides the best balance between spatial resolution and computational cost.

### Output: Seismic Zones

After training, each event $e_i$ is assigned to the seismic zone $S^k_z$ corresponding to its BMU neuron $c$. Neurons that attract many events represent **high-density seismicity hotspots** (major fault intersections, subduction interface patches). The resulting zones serve as spatially coherent partitions for the subsequent temporal clustering step.

In [ ]:
# =============================================================================
# CELL 5a — Feature Preparation and SOM Grid Size Selection
# =============================================================================

def prepare_spatial_features(df_in, feature_cols=None):
    """
    Extract and Min-Max normalise spatial features for SOM input.

    Normalisation to [0, 1] is essential because latitude, longitude,
    and depth are on very different numeric scales. Without normalisation,
    depth (0–700 km) would numerically dominate latitude differences
    (tenths of degrees), causing the SOM to zone purely by depth.

    Parameters
    ----------
    df_in       : DataFrame with spatial columns
    feature_cols: list of column names to use (default: lat, lon, depth)

    Returns
    -------
    X_norm   : np.ndarray, shape (N, 3), values in [0, 1]
    scaler   : fitted MinMaxScaler (for inverse-transform of prototypes)
    """
    if feature_cols is None:
        feature_cols = ['latitude', 'longitude', 'depth']
    X = df_in[feature_cols].values.astype(float)
    scaler = MinMaxScaler()
    X_norm = scaler.fit_transform(X)
    return X_norm, scaler


def quantization_error(som_model, data):
    """
    Compute mean Euclidean distance between each input and its BMU.
    This is the primary quality metric for SOM training.
    Lower QE = better representation of the data distribution.
    """
    errors = []
    for x in data:
        bmu = som_model.winner(x)
        errors.append(np.linalg.norm(x - som_model.get_weights()[bmu]))
    return np.mean(errors)


def select_som_grid(X_norm, candidates=None, n_iter=400, sigma_init=1.5, lr_init=0.1):
    """
    Train SOM models for a range of grid configurations and return
    quantization errors for each. The optimal grid is the one that
    minimises QE without being unnecessarily large.

    Grid size heuristic:
      - For N < 200   events: try 2×2 to 4×4
      - For N ~ 5000  events: try 5×5 to 7×7
      - For N > 10000 events: try 7×7 to 10×10

    Increasing grid size always reduces QE. Stop when the marginal
    improvement falls below 5% (elbow criterion).

    Parameters
    ----------
    X_norm    : normalised input array, shape (N, 3)
    candidates: list of (rows, cols) tuples to evaluate
    n_iter    : training iterations per candidate
    sigma_init: initial neighbourhood radius
    lr_init   : initial learning rate

    Returns
    -------
    results_df : DataFrame with columns ['grid', 'neurons', 'QE']
    """
    N = len(X_norm)
    if candidates is None:
        # Auto-generate sensible grid sizes based on catalog size
        max_side = max(2, min(8, int(np.sqrt(3 * np.sqrt(N)))))
        candidates = [(i, i) for i in range(2, max_side + 1)]
        candidates += [(i, i + 1) for i in range(2, max_side)]

    records = []
    for (gx, gy) in candidates:
        som_tmp = MiniSom(
            gx, gy, X_norm.shape[1],
            sigma=sigma_init,
            learning_rate=lr_init,
            neighborhood_function='gaussian',
            random_seed=RANDOM_SEED
        )
        som_tmp.random_weights_init(X_norm)
        som_tmp.train(X_norm, n_iter, verbose=False)
        qe = quantization_error(som_tmp, X_norm)
        records.append({'grid': (gx, gy), 'label': f'{gx}×{gy}',
                        'neurons': gx * gy, 'QE': qe})

    return pd.DataFrame(records)


# ── Prepare input features ────────────────────────────────────────────────────
X_spatial, spatial_scaler = prepare_spatial_features(df)
print(f"Input matrix: {X_spatial.shape}  (N events × 3 features)")
print(f"Features: latitude, longitude, depth  →  all normalised to [0, 1]")

# ── Grid search ───────────────────────────────────────────────────────────────
print("\nRunning SOM grid size search...")
grid_results = select_som_grid(X_spatial, n_iter=400)
print(grid_results[['label', 'neurons', 'QE']].to_string(index=False))

# Select optimal grid
best_row  = grid_results.loc[grid_results['QE'].idxmin()]
GRID_ROWS = int(best_row['grid'][0])
GRID_COLS = int(best_row['grid'][1])
print(f"\nSelected grid: {GRID_ROWS}×{GRID_COLS}  "
      f"({GRID_ROWS * GRID_COLS} neurons,  QE = {best_row['QE']:.5f})")
print("Override here if the elbow in the plot suggests a different size.")

In [ ]:
# =============================================================================
# CELL 5b — Train Final SOM
# =============================================================================

# ── Configurable parameters ───────────────────────────────────────────────────
# Override GRID_ROWS / GRID_COLS here if visual inspection of the QE plot
# suggests a different size than the automatic selection above.
SOM_GRID_X  = GRID_ROWS          # number of SOM rows
SOM_GRID_Y  = GRID_COLS          # number of SOM columns
SOM_ITER    = 600                 # total training iterations
SOM_SIGMA   = max(SOM_GRID_X, SOM_GRID_Y) / 2.0   # initial σ
SOM_LR      = 0.1                 # initial learning rate α₀
SNAP_EVERY  = 10                  # record QE every N iterations

print(f"Training SOM  {SOM_GRID_X}×{SOM_GRID_Y}  "
      f"|  iterations={SOM_ITER}  |  σ₀={SOM_SIGMA:.1f}  |  α₀={SOM_LR}")

som = MiniSom(
    SOM_GRID_X, SOM_GRID_Y,
    X_spatial.shape[1],
    sigma=SOM_SIGMA,
    learning_rate=SOM_LR,
    neighborhood_function='gaussian',
    random_seed=RANDOM_SEED
)
som.random_weights_init(X_spatial)

# ── Train with QE tracking ────────────────────────────────────────────────────
qe_history = []
for step in range(0, SOM_ITER, SNAP_EVERY):
    som.train(X_spatial, SNAP_EVERY, verbose=False)
    qe_history.append(quantization_error(som, X_spatial))

final_qe = quantization_error(som, X_spatial)
print(f"Training complete.  Final QE = {final_qe:.6f}")

# ── Assign each event to its SOM zone ────────────────────────────────────────
bmu_indices = [som.winner(x) for x in X_spatial]          # (row, col) for each event
zone_ids    = [r * SOM_GRID_Y + c for r, c in bmu_indices] # flatten to integer zone id
df['som_zone'] = zone_ids

n_active_zones = df['som_zone'].nunique()
print(f"Active seismic zones identified: {n_active_zones} "
      f"(of {SOM_GRID_X * SOM_GRID_Y} neurons)")
print("\nEvents per zone:")
print(df['som_zone'].value_counts().sort_index().to_string())

In [ ]:
# =============================================================================
# CELL 5c — SOM Visualisation
# =============================================================================

fig = plt.figure(figsize=(16, 4.5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# ── Panel A: QE convergence curve ────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])
iters = np.arange(len(qe_history)) * SNAP_EVERY
ax0.plot(iters, qe_history, color='steelblue', lw=2)
ax0.fill_between(iters, qe_history, alpha=0.15, color='steelblue')
ax0.set_xlabel('Training iteration')
ax0.set_ylabel('Quantization Error')
ax0.set_title('(a)  SOM Convergence')
ax0.grid(True, alpha=0.25)

# ── Panel B: SOM hit map (events per neuron) ──────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
hit_map = np.zeros((SOM_GRID_X, SOM_GRID_Y), dtype=int)
for r, c in bmu_indices:
    hit_map[r, c] += 1

im = ax1.imshow(hit_map, cmap='YlOrRd', aspect='auto', interpolation='nearest')
plt.colorbar(im, ax=ax1, label='Event count per neuron')

for i in range(SOM_GRID_X):
    for j in range(SOM_GRID_Y):
        ax1.text(j, i, str(hit_map[i, j]),
                 ha='center', va='center', fontsize=8.5, fontweight='bold')

ax1.set_xticks(range(SOM_GRID_Y))
ax1.set_yticks(range(SOM_GRID_X))
ax1.set_xlabel('SOM column')
ax1.set_ylabel('SOM row')
ax1.set_title('(b)  SOM Hit Map\n(event count per neuron)')

# ── Panel C: Epicentre map coloured by zone ───────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
n_zones = df['som_zone'].nunique()
cmap_z  = plt.cm.get_cmap('tab10', n_zones)

sc = ax2.scatter(
    df['longitude'], df['latitude'],
    c=df['som_zone'], cmap=cmap_z,
    s=45, alpha=0.85, edgecolors='k', linewidths=0.3
)

# Plot SOM prototype positions (neurons in geographic space)
weights_flat = som.get_weights().reshape(-1, 3)        # (n_neurons, 3)
protos_geo   = spatial_scaler.inverse_transform(weights_flat)  # back to degrees/km
ax2.scatter(
    protos_geo[:, 1], protos_geo[:, 0],  # lon, lat
    marker='*', s=180, c='blue', zorder=6,
    edgecolors='navy', linewidths=0.5, label='SOM prototype'
)

plt.colorbar(sc, ax=ax2, label='Zone ID')
ax2.set_xlabel('Longitude (°E)')
ax2.set_ylabel('Latitude (°N)')
ax2.set_title('(c)  Seismotectonic Zones\n(SOM prototype = blue star)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.2)

plt.suptitle('Stage 1 — Self-Organizing Map Results',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('figures/02_som_results.png', dpi=130, bbox_inches='tight')
plt.show()
print("Figure saved: figures/02_som_results.png")

---
## Cell 6: Stage 2 — Temporal DBSCAN (T-DBSCAN)

### Motivation for Temporal-Only Clustering within Zones

Having partitioned the catalog into spatially coherent seismotectonic zones using the SOM, the problem of declustering reduces to a **one-dimensional temporal problem** within each zone. Events in the same zone share a similar tectonic source; the question is simply: *are these events unusually close in time, suggesting a causal mainshock–aftershock relationship, or are they spread evenly in time, consistent with independent background loading?*

I use DBSCAN — Density-Based Spatial Clustering of Applications with Noise — applied **exclusively in the time dimension**. I call this T-DBSCAN to emphasise the temporal-only distance metric.

### Algorithm: T-DBSCAN

T-DBSCAN classifies every event in a zone into one of three categories based on two parameters:
- $\varepsilon$ (epsilon): the temporal neighbourhood radius in days
- $MinPts$: the minimum number of events required to form a dense cluster

**Definitions:**

**Core event:** Event $i$ is a core event if at least $MinPts$ events (including itself) have occurrence times within $\varepsilon$ days of $t_i$:

$$|\{j : |t_j - t_i| \leq \varepsilon\}| \geq MinPts$$

**Primary aftershock:** Event $j$ is a primary aftershock of core event $i$ if $|t_j - t_i| \leq \varepsilon$. It is directly density-reachable from $i$.

**Secondary aftershock:** Event $k$ is a secondary aftershock if there exists a chain of core events $i \to j \to \cdots \to k$ where each link is within $\varepsilon$ days. This captures the **cascading triggering** structure of real aftershock sequences — a large aftershock can itself generate further events.

**Background event:** Any event not density-reachable from any core event. In DBSCAN terminology this is a **noise point** — it does not belong to any dense temporal cluster.

**Cluster expansion procedure:** Starting from an unvisited core event, DBSCAN recursively expands the cluster by adding all events within $\varepsilon$ days, checking if any of those are also core events (and if so, adding their $\varepsilon$-neighbourhood), until no further expansion is possible. The final cluster contains the mainshock and its entire direct and indirect aftershock chain.

### Parameter Selection

**$\varepsilon$ (temporal radius):** I select $\varepsilon$ using the **k-nearest-neighbour distance plot**: for each event, I compute the distance to its $k$-th nearest neighbour in time. Sorting these distances in descending order and plotting them reveals an "elbow" — the natural scale at which inter-event times transition from clustered (short, left of elbow) to background-like (long, right of elbow). The elbow value is a data-driven estimate of $\varepsilon$.

**$MinPts$:** I test several values in the sensitivity analysis (Cell 12). For the main results I use $MinPts = 3$, appropriate for the density of this catalog. For the full NZ catalog with thousands of events per zone, $MinPts = 8$–$10$ is more suitable.

In [ ]:
# =============================================================================
# CELL 6a — T-DBSCAN Parameter Selection: k-NN Distance Plot
# =============================================================================

def knn_distance_plot(times, k=4, ax=None):
    """
    Build the k-th nearest-neighbour distance plot for ε selection.

    For 1D time series, sorting and differencing is equivalent to the
    k-NN search. The 'elbow' in the sorted k-distance curve separates
    the clustered (small distance) regime from the background (large
    distance) regime and provides a natural ε estimate.

    Parameters
    ----------
    times : array-like — event times in days
    k     : int        — which nearest neighbour to use (typically 2–5)
    ax    : matplotlib Axes or None

    Returns
    -------
    kth_dists : sorted k-NN distances (descending) for ε reading
    """
    t = np.sort(times).reshape(-1, 1)
    if len(t) <= k:
        return None

    nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto').fit(t)
    dists, _ = nbrs.kneighbors(t)
    kth_dists = np.sort(dists[:, -1])[::-1]   # sort descending

    if ax is not None:
        ax.plot(kth_dists, color='steelblue', lw=1.8)
        ax.set_xlabel('Events (sorted by decreasing distance)')
        ax.set_ylabel(f'{k}-NN distance  (days)')
        ax.set_title(f'k-NN Distance Plot  (k={k})\nElbow → suggested ε')
        ax.grid(True, alpha=0.25)

    return kth_dists


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Full catalog k-distance ───────────────────────────────────────────────────
k_val = min(4, len(df) - 1)
kd    = knn_distance_plot(df['time'].values, k=k_val, ax=axes[0])

if kd is not None:
    # Simple elbow detection: largest second derivative
    grad2 = np.abs(np.gradient(np.gradient(kd)))
    elbow_idx = int(np.argmax(grad2[2:-2])) + 2
    eps_suggested = float(kd[elbow_idx])
    axes[0].axhline(eps_suggested, color='crimson', ls='--', lw=1.8,
                    label=f'Suggested ε ≈ {eps_suggested:.3f} d')
    axes[0].legend(fontsize=9)
    print(f"k-NN elbow suggests ε ≈ {eps_suggested:.4f} days")

# ── QE vs grid size ───────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(range(len(grid_results)), grid_results['QE'],
        'o-', color='darkorange', ms=7, lw=2)
best_idx = grid_results['QE'].idxmin()
ax.plot(best_idx, grid_results.loc[best_idx, 'QE'],
        '*', ms=16, color='crimson',
        label=f"Best: {grid_results.loc[best_idx,'label']}")
ax.set_xticks(range(len(grid_results)))
ax.set_xticklabels(grid_results['label'], rotation=40, ha='right', fontsize=9)
ax.set_xlabel('SOM grid size')
ax.set_ylabel('Quantization Error (QE)')
ax.set_title('SOM Grid Selection via Quantization Error')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('figures/03_parameter_selection.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# CELL 6b — T-DBSCAN Core Implementation
# =============================================================================

def temporal_dbscan(times_sorted, eps_days, min_pts):
    """
    Temporal DBSCAN clustering on a one-dimensional time series.

    This function implements the full DBSCAN algorithm restricted to
    a single dimension (time). Because the input is sorted, the ε-
    neighbourhood query reduces to a binary search (O(log N)), making
    the overall complexity O(N log N) rather than the O(N²) of a
    naive all-pairs comparison.

    Parameters
    ----------
    times_sorted : array-like   — event times in days, must be sorted
    eps_days     : float        — ε: temporal neighbourhood radius (days)
    min_pts      : int          — MinPts: minimum events for a core point

    Returns
    -------
    labels : np.ndarray of int
        -1  → Background (BG): noise / non-clustered event
         0  → Cluster 0  (first aftershock sequence)
         1  → Cluster 1  (second aftershock sequence)
         ... etc.

    Notes
    -----
    Border events (within ε of a core event but not core themselves)
    are assigned to the cluster of the first core point that reached
    them. This matches the original DBSCAN convention.
    """
    times   = np.asarray(times_sorted, dtype=float)
    n       = len(times)
    labels  = np.full(n, -1, dtype=np.int32)   # default: unassigned (BG)
    visited = np.zeros(n, dtype=bool)
    cluster_id = 0

    def eps_neighbourhood(idx):
        """Return indices of all events within ε days of times[idx]."""
        t_i = times[idx]
        lo  = int(np.searchsorted(times, t_i - eps_days, side='left'))
        hi  = int(np.searchsorted(times, t_i + eps_days, side='right'))
        return list(range(lo, hi))

    for i in range(n):

        if visited[i]:
            continue
        visited[i] = True

        neighbourhood = eps_neighbourhood(i)

        # ── Event i is NOT a core point → background ──────────────────────────
        if len(neighbourhood) < min_pts:
            labels[i] = -1      # remains background
            continue

        # ── Event i IS a core point → start new cluster ───────────────────────
        labels[i] = cluster_id
        seed_queue = set(neighbourhood) - {i}   # events to expand from

        while seed_queue:
            j = seed_queue.pop()

            if not visited[j]:
                visited[j] = True
                j_neighbours = eps_neighbourhood(j)

                # If j is also a core point, expand the cluster further
                if len(j_neighbours) >= min_pts:
                    seed_queue |= set(j_neighbours)

            # Assign j to current cluster if not yet in any cluster
            if labels[j] < 0:
                labels[j] = cluster_id

        cluster_id += 1

    return labels


# ── Set final T-DBSCAN parameters ────────────────────────────────────────────
# Adjust these based on the k-NN distance plot above and the
# sensitivity analysis in Cell 12.
#
# eps_days = 1.0 is a common starting point (one day window).
# For sparse catalogs, try 0.5–2.0.
# min_pts  = 3 is appropriate for this small sample.
# For the full NZ catalog use min_pts = 5–10.
EPS_DAYS = 1.0
MIN_PTS  = 3

print(f"T-DBSCAN parameters set:")
print(f"  ε (temporal radius) = {EPS_DAYS} day(s)")
print(f"  MinPts              = {MIN_PTS}")

In [ ]:
# =============================================================================
# CELL 6c — Apply T-DBSCAN Zone-by-Zone and Collect Results
# =============================================================================

# Initialise output columns
df['cluster_in_zone']  = -1    # local cluster id within SOM zone
df['cluster_global']   = -1    # globally unique cluster id across all zones
df['event_type']       = 'BG'  # default classification: background

global_offset   = 0
zone_records    = []

for zone_id in sorted(df['som_zone'].unique()):

    zone_mask = df['som_zone'] == zone_id
    zone_df   = df[zone_mask].sort_values('time')

    if len(zone_df) < 2:
        # Single event in zone — cannot form a cluster, remains BG
        continue

    # Run T-DBSCAN on the zone's sorted time series
    labels = temporal_dbscan(
        zone_df['time'].values,
        eps_days=EPS_DAYS,
        min_pts=MIN_PTS
    )

    # Map results back to the main dataframe
    for local_idx, (orig_idx, _) in enumerate(zone_df.iterrows()):
        lbl = int(labels[local_idx])
        df.at[orig_idx, 'cluster_in_zone'] = lbl

        if lbl >= 0:   # event belongs to a temporal cluster → aftershock
            df.at[orig_idx, 'cluster_global'] = lbl + global_offset
            df.at[orig_idx, 'event_type']     = 'AF'
        # else: remains 'BG'

    n_clusters_zone = int(labels.max()) + 1 if labels.max() >= 0 else 0
    n_af_zone       = int(np.sum(labels >= 0))
    n_bg_zone       = int(np.sum(labels < 0))
    global_offset  += n_clusters_zone

    zone_records.append({
        'Zone': zone_id,
        'N_events'  : len(zone_df),
        'N_AF'      : n_af_zone,
        'N_BG'      : n_bg_zone,
        'N_clusters': n_clusters_zone
    })

zone_summary = pd.DataFrame(zone_records)

# ── Global totals ─────────────────────────────────────────────────────────────
N_TOTAL    = len(df)
N_AF       = (df['event_type'] == 'AF').sum()
N_BG       = (df['event_type'] == 'BG').sum()
N_CLUSTERS = max(0, int(df['cluster_global'].max()) + 1)

print("Per-zone declustering results:")
print(zone_summary.to_string(index=False))
print()
print("═" * 42)
print("GLOBAL DECLUSTERING SUMMARY")
print("═" * 42)
print(f"Total events       : {N_TOTAL}")
print(f"Aftershocks   (AF) : {N_AF}  ({100*N_AF/N_TOTAL:.1f}%)")
print(f"Background    (BG) : {N_BG}  ({100*N_BG/N_TOTAL:.1f}%)")
print(f"Clusters found     : {N_CLUSTERS}")

---
## Cell 7: Results Visualisation

I visualise the declustering output using four complementary plots. Together they provide a spatial and temporal characterisation of the two event populations.

**Epicentre map:** Aftershocks (blue) concentrate around the major seismically active fault regions, while background events (grey) are more evenly distributed along fault boundaries — reflecting their tectonic loading origin rather than triggered clustering.

**Cumulative event count:** The total event count (red) shows step-like jumps at times of major seismic activity (aftershock bursts). The cumulative AF count (blue) tracks these jumps, while the BG count (black) should increase at a **roughly constant rate** — consistent with a stationary Poisson process.

**Lambda (λ) rate plot:** Shows the number of events per time bin. AF peaks correspond to mainshock–aftershock sequences; BG shows a flat rate without peaks.

**Magnitude–time plot:** Aftershocks typically occur in swarms of smaller magnitudes immediately after a larger event (Bath's law: largest aftershock ≈ Mmain − 1.2). Background events have a more uniform magnitude distribution over time.

In [ ]:
# =============================================================================
# CELL 7 — Four-Panel Results Visualisation
# =============================================================================

af_df = df[df['event_type'] == 'AF'].copy()
bg_df = df[df['event_type'] == 'BG'].copy()

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.30)

# ── Panel A: Epicentre Map ────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])

ax0.scatter(bg_df['longitude'], bg_df['latitude'],
            c='#cccccc', s=40, alpha=0.95,
            edgecolors='#888888', linewidths=0.3,
            label=f'Background  (n = {len(bg_df)})', zorder=2)

# Colour AF points by cluster id for visual cluster identity
if len(af_df) > 0:
    sc_af = ax0.scatter(af_df['longitude'], af_df['latitude'],
                        c=af_df['cluster_global'],
                        cmap='tab10', s=55, alpha=0.9,
                        edgecolors='navy', linewidths=0.3,
                        label=f'Aftershocks  (n = {len(af_df)})', zorder=3)

ax0.set_xlabel('Longitude (°E)')
ax0.set_ylabel('Latitude (°N)')
ax0.set_title('(a)  Epicentre Map\nAF = coloured by cluster, BG = grey')
ax0.legend(fontsize=9, loc='lower left')
ax0.grid(True, alpha=0.2)

# ── Panel B: Cumulative Count ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])

def cumulative_line(times, ax, color, label, lw=2):
    t = np.sort(times)
    if len(t) > 0:
        ax.plot(t, np.arange(1, len(t)+1), color=color, lw=lw, label=label)

cumulative_line(df['time'],    ax1, '#e74c3c', f'Total  (n={N_TOTAL})', lw=2)
cumulative_line(af_df['time'], ax1, '#2980b9', f'AF  (n={N_AF})',    lw=1.8)
cumulative_line(bg_df['time'], ax1, '#2c3e50', f'BG  (n={N_BG})',    lw=1.8)

ax1.set_xlabel('Time  (days from catalog start)')
ax1.set_ylabel('Cumulative event count')
ax1.set_title('(b)  Cumulative Event Count')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.2)

# ── Panel C: Lambda Rate Plot ─────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])

n_bins = max(6, N_TOTAL // 2)
t_min  = df['time'].min()
t_max  = df['time'].max()
bin_edges = np.linspace(t_min, t_max, n_bins + 1)
centers   = (bin_edges[:-1] + bin_edges[1:]) / 2

for data_t, clr, lbl in [
    (df['time'],    '#e74c3c', f'Total'),
    (af_df['time'], '#2980b9', 'AF'),
    (bg_df['time'], '#2c3e50', 'BG')
]:
    if len(data_t) > 0:
        cnts, _ = np.histogram(data_t, bins=bin_edges)
        ax2.plot(centers, cnts, 'o-', color=clr, ms=4, lw=1.6, label=lbl)

ax2.set_xlabel('Time  (days)')
ax2.set_ylabel('Event count per bin')
ax2.set_title('(c)  Event Rate  λ(t)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.2)

# ── Panel D: Magnitude–Time ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])

ax3.scatter(bg_df['time'], bg_df['magnitude'],
            c='#cccccc', s=35, alpha=0.9,
            edgecolors='#888888', lw=0.3, label='BG', zorder=2)

if len(af_df) > 0:
    ax3.scatter(af_df['time'], af_df['magnitude'],
                c='#2980b9', s=50, alpha=0.9,
                edgecolors='navy', lw=0.3, label='AF', zorder=3)

ax3.axhline(Mc, color='crimson', ls='--', lw=1.5, label=f'Mc = {Mc:.1f}')
ax3.set_xlabel('Time  (days)')
ax3.set_ylabel('Magnitude')
ax3.set_title('(d)  Magnitude–Time Distribution')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.2)

plt.suptitle(
    'SOM–DBSCAN Declustering Results — New Zealand Earthquake Catalog',
    fontsize=13, fontweight='bold', y=1.01
)
plt.savefig('figures/04_declustering_results.png', dpi=130, bbox_inches='tight')
plt.show()
print("Figure saved: figures/04_declustering_results.png")

---
## Cell 8: Validation — Coefficient of Variation (COV)

### Theory

The **Coefficient of Variation (COV)** of inter-event times (IET) is a dimensionless measure of temporal clustering. For a time series of $N$ events occurring at times $t_1 < t_2 < \cdots < t_N$, the inter-event times are:

$$\Delta t_j = t_{j+1} - t_j \quad j = 1, 2, \ldots, N-1$$

The COV is the ratio of the standard deviation to the mean of the IET sequence:

$$\text{COV} = \frac{\sigma(\Delta t)}{\mu(\Delta t)} = \frac{\sqrt{E[(\Delta t)^2] - [E(\Delta t)]^2}}{E(\Delta t)}$$

### Physical Interpretation

The COV is directly related to the statistical nature of the point process:

| COV | Process type | Expected for |
|-----|-------------|----------------|
| COV = 0 | Periodic (perfectly regular) | — |
| 0 < COV < 1 | Sub-Poisson (quasi-regular) | — |
| COV = 1 | Poisson process (exponential IET distribution) | **Background seismicity** |
| COV > 1 | Super-Poisson (bursty / power-law IET) | **Aftershock sequences** |

A well-declustered catalog should produce:

$$\text{COV}_{\text{AF}} > \text{COV}_{\text{Total}} > \text{COV}_{\text{BG}} \approx 1$$

The near-unity COV for BGs confirms that the background seismicity approximates a Poisson process, validating the declustering. The high COV for AFs reflects the rapid succession of events in an aftershock burst followed by long quiet periods — a hallmark of Omori-law decay.

In [ ]:
# =============================================================================
# CELL 8 — Coefficient of Variation Validation
# =============================================================================

def coefficient_of_variation(times):
    """
    Compute the Coefficient of Variation of inter-event times.

    Returns nan for fewer than 2 events (IET is undefined).
    COV ≈ 1 → Poisson background
    COV >> 1 → temporally clustered (aftershock-like)
    """
    t = np.sort(np.asarray(times, dtype=float))
    if len(t) < 2:
        return np.nan, np.array([])

    iet  = np.diff(t)
    mu   = np.mean(iet)
    sigma = np.std(iet, ddof=1)   # sample standard deviation

    if mu < 1e-12:   # degenerate case: all events simultaneous
        return np.nan, iet

    return sigma / mu, iet


cov_T, iet_T = coefficient_of_variation(df['time'])
cov_A, iet_A = coefficient_of_variation(af_df['time'])
cov_B, iet_B = coefficient_of_variation(bg_df['time'])

# ── Print results ─────────────────────────────────────────────────────────────
print("═" * 50)
print("COEFFICIENT OF VARIATION — VALIDATION")
print("═" * 50)
print(f"  COV (Total catalog) = {cov_T:.4f}")
print(f"  COV (Aftershocks)   = {cov_A:.4f}   [expected > 1]")
print(f"  COV (Background)    = {cov_B:.4f}   [expected ≈ 1]")
print()

for label, cov in [('Total', cov_T), ('AF', cov_A), ('BG', cov_B)]:
    if np.isnan(cov):
        note = 'insufficient data'
    elif cov < 0.85:
        note = 'sub-Poisson (quasi-regular)'
    elif cov <= 1.15:
        note = '≈ Poisson   ✓  (random / background-like)'
    else:
        note = 'super-Poisson ✓  (clustered / aftershock-like)'
    print(f"  {label:8s}: {cov:.4f}  →  {note}")

# ── Hierarchy check ───────────────────────────────────────────────────────────
print()
if not (np.isnan(cov_A) or np.isnan(cov_B)):
    if cov_A > cov_B:
        print("✓  COV_AF > COV_BG confirmed — AFs are more temporally clustered than BGs.")
    else:
        print("⚠  COV_AF ≤ COV_BG — consider adjusting ε or MinPts.")
        print("   Small catalogs may not have enough events per zone for reliable clustering.")

# ── IET histogram comparison ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, iet, label, color, cov_val in [
    (axes[0], iet_T, f'Total\nCOV = {cov_T:.3f}', '#e74c3c', cov_T),
    (axes[1], iet_A, f'Aftershocks\nCOV = {cov_A:.3f}', '#2980b9', cov_A),
    (axes[2], iet_B, f'Background\nCOV = {cov_B:.3f}', '#2c3e50', cov_B),
]:
    if len(iet) > 0:
        ax.hist(iet, bins=min(20, max(5, len(iet)//2)),
                color=color, alpha=0.65, edgecolor='white')
        ax.axvline(np.mean(iet), color='black', ls='--', lw=1.5,
                   label=f'Mean = {np.mean(iet):.3f} d')
        ax.set_title(label)
        ax.set_xlabel('Inter-event time  (days)')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.2)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=11)
        ax.set_title(label)

plt.suptitle('Inter-Event Time Distributions and COV Values',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/05_cov_validation.png', dpi=130, bbox_inches='tight')
plt.show()

---
## Cell 9: IET vs IED Scatter Analysis

### Rationale

The joint distribution of **Inter-Event Time (IET)** and **Inter-Event Distance (IED)** for consecutive pairs of events in the catalog provides a two-dimensional view of the separation between the two populations.

Aftershock events follow the parent mainshock closely in **both space and time**: they nucleate on or near the same fault segment (small IED) within hours to days of the mainshock (small IET). Background events, by contrast, nucleate on different parts of the fault system separated in time by the slow accumulation of tectonic stress (large IET, large IED).

On a **log–log plot** of IET vs IED, a well-declustered catalog should show two visually distinct populations:
- **AFs (red):** Concentrated in the lower-left quadrant (short time, short distance)
- **BGs (blue):** Spread across the upper-right region (long time, large distance)

The absence of overlap between these two clouds confirms that the SOM–DBSCAN method has cleanly separated the two populations.

In [ ]:
# =============================================================================
# CELL 9 — IET vs IED Log-Log Scatter
# =============================================================================

def compute_consecutive_iet_ied(df_in):
    """
    For each consecutive pair of events (sorted by time), compute:
      - IET: inter-event time in days
      - IED: inter-event 3D hypocenter distance in km
      - event_type: label of the second event in the pair

    This follows the pair-based approach standard in seismicity analysis.
    For large catalogs, computing ALL pairs (not just consecutive) is
    more informative but computationally expensive — see the note below.
    """
    df_s = df_in.sort_values('time').reset_index(drop=True)
    iet_list, ied_list, type_list = [], [], []

    for i in range(len(df_s) - 1):
        dt = df_s.loc[i+1, 'time'] - df_s.loc[i, 'time']   # days
        dd = hypocenter_distance_km(
            df_s.loc[i,   'latitude'], df_s.loc[i,   'longitude'], df_s.loc[i,   'depth'],
            df_s.loc[i+1, 'latitude'], df_s.loc[i+1, 'longitude'], df_s.loc[i+1, 'depth']
        )   # km
        iet_list.append(dt)
        ied_list.append(dd)
        type_list.append(df_s.loc[i+1, 'event_type'])

    return np.array(iet_list), np.array(ied_list), np.array(type_list)


iet_vals, ied_vals, pair_types = compute_consecutive_iet_ied(df)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: coloured by event type
ax = axes[0]
for etype, color, label, marker in [
    ('BG', '#2980b9', 'Background  (BG)', 'o'),
    ('AF', '#e74c3c', 'Aftershock  (AF)', '^')
]:
    msk = pair_types == etype
    if msk.sum() > 0:
        ax.scatter(iet_vals[msk], ied_vals[msk],
                   c=color, s=30, alpha=0.7, marker=marker, label=label)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Inter-Event Time  Δt  (days)')
ax.set_ylabel('Inter-Event Distance  Δd  (km)')
ax.set_title('(a)  IET vs IED — log-log\nColoured by declustering label')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.2)

# Panel B: 2D histogram density view
ax = axes[1]
valid = (iet_vals > 0) & (ied_vals > 0)
if valid.sum() > 2:
    h = ax.hist2d(
        np.log10(iet_vals[valid]),
        np.log10(ied_vals[valid]),
        bins=10, cmap='YlOrRd'
    )
    plt.colorbar(h[3], ax=ax, label='Count')
ax.set_xlabel('log₁₀(IET)  (days)')
ax.set_ylabel('log₁₀(IED)  (km)')
ax.set_title('(b)  IET vs IED — 2D density\n(all event pairs)')
ax.grid(True, alpha=0.2)

plt.suptitle('Inter-Event Time vs Inter-Event Distance Analysis',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/06_iet_ied.png', dpi=130, bbox_inches='tight')
plt.show()
print("Figure saved: figures/06_iet_ied.png")

---
## Cell 10: Spatial Clustering Validation — m-Morisita Index

### Theory

The **m-Morisita index** is a multi-scale, non-parametric measure of spatial clustering. It quantifies whether a set of point locations is more aggregated (clustered), more dispersed, or indistinguishable from a random Poisson distribution — at a range of spatial scales $\delta$.

For a point pattern of $N$ events partitioned into $Q$ quadrats (grid cells) of size $\delta$, with $n_i$ events in quadrat $i$, the **m-point Morisita index** is:

$$I_{m,\delta} = Q^{m-1} \cdot \frac{\displaystyle\sum_{i=1}^{Q} n_i(n_i-1)(n_i-2)\cdots(n_i-m+1)}{N(N-1)(N-2)\cdots(N-m+1)}$$

where $m \geq 2$ is the order of the index.

### Interpretation

- $I_{m,\delta} = 1$: events are randomly distributed (Poisson) — **expected for background seismicity**
- $I_{m,\delta} > 1$: events are spatially clustered — **expected for aftershock sequences**
- $I_{m,\delta} < 1$: events are more dispersed than random

### Multi-Scale Analysis

By evaluating $I_{m,\delta}$ across a logarithmically spaced range of $\delta$ values and plotting $\log(I_{m,\delta})$ against $\log(\delta)$, I obtain the **m-Morisita slope spectrum**. For multifractal point patterns (as observed in seismicity), the index follows a power law:

$$I_{m,\delta} \propto \delta^{(m-1)(D_m - E)}$$

where $E = 2$ (for 2D spatial analysis) and $D_m$ is the generalised fractal dimension of order $m$. The slope of the $\log(I)$–$\log(\delta)$ plot gives $S_m = -(m-1)(D_m - E)$.

**Expected result:** $I_{m,\delta}^{\text{AF}} > I_{m,\delta}^{\text{Total}} > I_{m,\delta}^{\text{BG}}$ at every scale $\delta$ and for all $m$.

In [ ]:
# =============================================================================
# CELL 10 — m-Morisita Spatial Clustering Index
# =============================================================================

def morisita_index_2d(lats, lons, m, delta_deg):
    """
    Compute the m-point 2D Morisita index for a geographic point pattern.

    The study region is partitioned into a regular grid of square quadrats
    with side length delta_deg (in degrees). Events are counted per quadrat.
    The index is then computed from the quadrat counts.

    Parameters
    ----------
    lats, lons : array-like — geographic coordinates (decimal degrees)
    m          : int        — order of index (2, 3, 4, or 5)
    delta_deg  : float      — quadrat side length in degrees

    Returns
    -------
    Im : float — m-Morisita index value (NaN if insufficient data)
    """
    lats = np.asarray(lats, dtype=float)
    lons = np.asarray(lons, dtype=float)
    N = len(lats)

    if N < m:
        return np.nan

    # Build grid covering the point cloud with margin
    lat_edges = np.arange(
        np.floor(lats.min() / delta_deg) * delta_deg,
        np.ceil(lats.max()  / delta_deg) * delta_deg + delta_deg,
        delta_deg
    )
    lon_edges = np.arange(
        np.floor(lons.min() / delta_deg) * delta_deg,
        np.ceil(lons.max()  / delta_deg) * delta_deg + delta_deg,
        delta_deg
    )

    Q_rows = len(lat_edges) - 1
    Q_cols = len(lon_edges) - 1
    Q = Q_rows * Q_cols

    if Q < 1:
        return np.nan

    H, _, _ = np.histogram2d(lats, lons, bins=[lat_edges, lon_edges])
    n_i = H.flatten().astype(float)

    # Falling factorial: x * (x-1) * ... * (x-m+1)
    def fall_fact(x_arr, order):
        result = np.ones_like(x_arr)
        for k in range(order):
            result *= np.maximum(x_arr - k, 0.0)
        return result

    numerator   = (Q ** (m - 1)) * np.sum(fall_fact(n_i, m))
    denominator = fall_fact(np.array([float(N)]), m)[0]

    if denominator < 1e-10:
        return np.nan

    return numerator / denominator


def morisita_spectrum(lats, lons, m_orders=(2, 3, 4, 5), n_scales=15):
    """
    Compute the multi-scale Morisita spectrum for orders m = 2, 3, 4, 5.
    Returns log10(δ) and log10(Im,δ) arrays for plotting.
    """
    lats, lons = np.asarray(lats), np.asarray(lons)
    lat_range  = lats.max() - lats.min()
    lon_range  = lons.max() - lons.min()
    max_delta  = min(lat_range, lon_range) / 1.5 + 1e-6
    min_delta  = max(lat_range, lon_range) / (8 * n_scales) + 1e-6

    deltas  = np.logspace(np.log10(min_delta), np.log10(max_delta), n_scales)
    results = {}

    for m in m_orders:
        im_vals = np.array([morisita_index_2d(lats, lons, m, d) for d in deltas])
        results[m] = (deltas, im_vals)

    return results


# ── Compute for Total, AF, BG ─────────────────────────────────────────────────
datasets = {
    'Total (TE)': (df['latitude'].values,    df['longitude'].values),
    'AFs'       : (af_df['latitude'].values, af_df['longitude'].values),
    'BGs'       : (bg_df['latitude'].values, bg_df['longitude'].values),
}

m_colors = {2: '#e74c3c', 3: '#2c3e50', 4: '#2980b9', 5: '#27ae60'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (title, (lats_s, lons_s)) in zip(axes, datasets.items()):
    if len(lats_s) < 4:
        ax.text(0.5, 0.5, f'Insufficient data\n(n={len(lats_s)})',
                ha='center', va='center', transform=ax.transAxes, fontsize=10)
        ax.set_title(title)
        continue

    spec = morisita_spectrum(lats_s, lons_s)

    for m, (deltas, im_vals) in spec.items():
        valid = ~np.isnan(im_vals) & (im_vals > 0)
        if valid.sum() < 2:
            continue
        ax.plot(
            np.log10(deltas[valid]),
            np.log10(im_vals[valid]),
            'o-', color=m_colors[m], ms=4, lw=1.5,
            label=f'm = {m}'
        )

    ax.axhline(0, color='grey', ls=':', lw=1, label='Im = 1 (Poisson)')
    ax.set_xlabel('log₁₀(δ)  — quadrat size (degrees)')
    ax.set_ylabel('log₁₀(Im,δ)')
    ax.set_title(f'm-Morisita Index\n{title}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle(
    'Multi-scale m-Morisita Spatial Clustering Index\n'
    'Expected: Im(AF) > Im(Total) > Im(BG) at all scales',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('figures/07_morisita.png', dpi=130, bbox_inches='tight')
plt.show()
print("Figure saved: figures/07_morisita.png")

---
## Cell 11: Cross-Validation Against ETAS Model Outputs

This is an **original contribution** of my thesis analysis, going beyond the paper's validation framework.

The earthquake catalog was previously processed using an **Epidemic Type Aftershock Sequence (ETAS)** model, which provides probabilistic estimates of the aftershock character of each event. These are completely independent of the SOM–DBSCAN clustering — the ETAS model makes no use of spatial zones or density-based grouping.

By comparing the SOM–DBSCAN binary labels (AF / BG) against the continuous ETAS-derived quantities, I can assess whether the geometric/density-based approach captures the same physics as the branching-process model:

- **`R+` (ETAS branching ratio):** The conditional probability that this event was triggered by a preceding event. High R+ → the event is aftershock-like. SOM–DBSCAN AFs should have systematically higher R+.
- **`n_parent`:** Expected number of ancestor events in the ETAS branching tree. Events deep in an aftershock cascade (large n_parent) should be classified as AF.
- **`n_child`:** Expected number of future offspring. Large mainshocks at the start of a cluster have high n_child and may be classified as BG (the mainshock itself) or AF (if following another event).
- **`dm+`:** Magnitude difference from the nearest parent event. Most aftershocks are smaller than their trigger (dm+ > 0), following Bath's law.

In [ ]:
# =============================================================================
# CELL 11 — Cross-Validation with ETAS Outputs
# =============================================================================

etas_features = ['R+', 'N+', 'n_child', 'n_parent', 'dm+']
etas_available = [f for f in etas_features if f in df.columns]

# ── Numerical comparison ──────────────────────────────────────────────────────
print("Mean ETAS feature values by SOM–DBSCAN label:")
print(df.groupby('event_type')[etas_available].mean().round(5).to_string())
print()

# Interpret R+ comparison
if 'R+' in etas_available:
    r_af = df[df['event_type']=='AF']['R+'].mean()
    r_bg = df[df['event_type']=='BG']['R+'].mean()
    if not (np.isnan(r_af) or np.isnan(r_bg)):
        if r_af > r_bg:
            print(f"✓  Mean R+ (AF) = {r_af:.4f}  >  Mean R+ (BG) = {r_bg:.4f}")
            print("   SOM–DBSCAN AFs have higher ETAS branching ratio — consistent.")
        else:
            print(f"⚠  Mean R+ (AF) = {r_af:.4f}  ≤  Mean R+ (BG) = {r_bg:.4f}")
            print("   May reflect small-sample noise or parameter sensitivity.")

# ── Visualisation ─────────────────────────────────────────────────────────────
n_feats = len(etas_available)
fig, axes = plt.subplots(1, min(n_feats, 3), figsize=(14, 4.5))
if n_feats == 1:
    axes = [axes]

plot_pairs = etas_available[:3]
palette    = {'AF': '#2980b9', 'BG': '#cccccc'}

for ax, feat in zip(axes, plot_pairs):
    # Violin + strip for distribution comparison
    for i, etype in enumerate(['BG', 'AF']):
        sub = df[df['event_type'] == etype][feat].dropna()
        if len(sub) > 0:
            parts = ax.violinplot(sub, positions=[i], widths=0.6,
                                  showmedians=True, showextrema=False)
            for pc in parts['bodies']:
                pc.set_facecolor(palette[etype])
                pc.set_alpha(0.6)
            ax.scatter(
                np.random.normal(i, 0.06, size=len(sub)),
                sub.values,
                c=palette[etype], s=20, alpha=0.7, zorder=3
            )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['BG', 'AF'])
    ax.set_ylabel(feat)
    ax.set_title(f'ETAS  {feat}\nby SOM–DBSCAN label')
    ax.grid(True, axis='y', alpha=0.2)

plt.suptitle(
    'Cross-Validation: SOM–DBSCAN Labels vs ETAS Branching Statistics',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('figures/08_etas_crossval.png', dpi=130, bbox_inches='tight')
plt.show()
print("Figure saved: figures/08_etas_crossval.png")

---
## Cell 12: Parameter Sensitivity Analysis

A key methodological concern in any declustering study is the **sensitivity of results to parameter choices**. If the fraction of events classified as aftershocks changes drastically with small changes in $\varepsilon$ or $MinPts$, the declustered catalog is unreliable.

I systematically vary both parameters and report the resulting AF count, BG count, number of clusters, and COV of the background seismicity. The **target region** is:
- COV_BG close to 1.0 (Poisson background)
- A physically plausible AF fraction (typically 30–70% for active seismic regions)
- Stable results across a neighbourhood of parameter values

The optimal parameter combination is selected from this table for the final declustered catalog.

In [ ]:
# =============================================================================
# CELL 12 — Parameter Sensitivity Analysis
# =============================================================================

def run_declustering(df_in, eps, minpts):
    """Run the full T-DBSCAN step for a given (eps, minpts) pair."""
    df_tmp = df_in.copy()
    df_tmp['event_type'] = 'BG'

    g_offset = 0
    for zid in sorted(df_tmp['som_zone'].unique()):
        zm  = df_tmp['som_zone'] == zid
        zdf = df_tmp[zm].sort_values('time')
        if len(zdf) < 2:
            continue
        lbs = temporal_dbscan(zdf['time'].values, eps, minpts)
        for li, (oi, _) in enumerate(zdf.iterrows()):
            if lbs[li] >= 0:
                df_tmp.at[oi, 'event_type'] = 'AF'
        n_cl = int(lbs.max()) + 1 if lbs.max() >= 0 else 0
        g_offset += n_cl

    n_af = (df_tmp['event_type'] == 'AF').sum()
    n_bg = (df_tmp['event_type'] == 'BG').sum()
    cov_bg, _ = coefficient_of_variation(
        df_tmp[df_tmp['event_type'] == 'BG']['time']
    )
    return n_af, n_bg, g_offset, cov_bg


eps_range  = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0]
mpts_range = [2, 3, 4, 5]

print(f"{'ε (days)':>10}  {'MinPts':>7}  {'N_AF':>6}  {'N_BG':>6}  "
      f"{'Clusters':>9}  {'COV_BG':>8}  {'AF%':>6}")
print("─" * 60)

sens_records = []
for eps_t in eps_range:
    for mpt in mpts_range:
        n_af, n_bg, n_cl, cov_bg_val = run_declustering(df, eps_t, mpt)
        af_pct = 100.0 * n_af / len(df) if len(df) > 0 else 0
        cov_str = f"{cov_bg_val:.3f}" if not np.isnan(cov_bg_val) else '  nan'
        print(f"{eps_t:>10.2f}  {mpt:>7d}  {n_af:>6d}  {n_bg:>6d}  "
              f"{n_cl:>9d}  {cov_str:>8}  {af_pct:>5.1f}%")
        sens_records.append({
            'eps': eps_t, 'min_pts': mpt,
            'N_AF': n_af, 'N_BG': n_bg,
            'N_clusters': n_cl,
            'COV_BG': cov_bg_val,
            'AF_pct': af_pct
        })

print("─" * 60)
print("Target: COV_BG ≈ 1.0 and stable AF% across neighbouring ε/MinPts values.")

sens_df = pd.DataFrame(sens_records)
sens_df.to_csv('outputs/sensitivity_analysis.csv', index=False)
print("Sensitivity table saved: outputs/sensitivity_analysis.csv")

---
## Cell 13: Adaptive-ε Extension

One limitation of using a global fixed $\varepsilon = 1$ day is that seismicity rates vary substantially across the New Zealand catalog. In regions of high background seismicity (e.g., the Hikurangi subduction interface), events occur more frequently and a shorter $\varepsilon$ is appropriate to avoid false cluster assignments. In quieter zones, a larger $\varepsilon$ may be needed to capture genuine aftershock sequences.

I implement an **adaptive-ε scheme** that estimates a zone-specific temporal radius from the inter-event time distribution within each SOM zone. Specifically, I set $\varepsilon_k$ equal to the $p$-th percentile of inter-event times in zone $k$:

$$\varepsilon_k = \text{percentile}_p\left\{\Delta t_j : \text{event } j \in S^k_z\right\}$$

Using a low percentile (e.g., $p = 25$) ensures that $\varepsilon_k$ captures the timescale of genuine aftershock clustering within each zone rather than the background inter-event time. This approach is a novel extension that I contribute beyond the standard fixed-ε methodology.

In [ ]:
# =============================================================================
# CELL 13 — Adaptive-ε T-DBSCAN (My Original Extension)
# =============================================================================

def adaptive_temporal_dbscan(df_in, eps_percentile=25, min_pts=3,
                              eps_min=0.1, eps_max=5.0):
    """
    Run T-DBSCAN with a zone-specific adaptive ε estimated from the
    inter-event time distribution within each SOM zone.

    Parameters
    ----------
    df_in          : DataFrame with 'som_zone', 'time' columns
    eps_percentile : int    — IET percentile used for ε estimation (default 25)
    min_pts        : int    — minimum events for a core point
    eps_min        : float  — minimum allowed ε (days), prevents ε→0
    eps_max        : float  — maximum allowed ε (days), prevents ε→∞

    Returns
    -------
    df_out    : DataFrame with 'event_type_adaptive' column
    zone_eps  : dict mapping zone_id → ε used
    """
    df_out = df_in.copy()
    df_out['event_type_adaptive'] = 'BG'
    zone_eps = {}

    for zone_id in sorted(df_out['som_zone'].unique()):
        zone_mask = df_out['som_zone'] == zone_id
        zone_df   = df_out[zone_mask].sort_values('time')

        if len(zone_df) < 2:
            continue

        # Estimate zone-specific ε from inter-event time distribution
        iet_zone = np.diff(zone_df['time'].values)
        if len(iet_zone) == 0:
            continue

        eps_z = float(np.percentile(iet_zone, eps_percentile))
        eps_z = np.clip(eps_z, eps_min, eps_max)
        zone_eps[zone_id] = eps_z

        labels = temporal_dbscan(zone_df['time'].values,
                                  eps_days=eps_z, min_pts=min_pts)

        for local_i, (orig_idx, _) in enumerate(zone_df.iterrows()):
            if labels[local_i] >= 0:
                df_out.at[orig_idx, 'event_type_adaptive'] = 'AF'

    return df_out, zone_eps


# ── Run adaptive version ──────────────────────────────────────────────────────
df_adaptive, zone_eps_dict = adaptive_temporal_dbscan(
    df, eps_percentile=25, min_pts=MIN_PTS
)

n_af_adap = (df_adaptive['event_type_adaptive'] == 'AF').sum()
n_bg_adap = (df_adaptive['event_type_adaptive'] == 'BG').sum()

print("Adaptive ε values per zone:")
for z, e in sorted(zone_eps_dict.items()):
    n_in_zone = (df['som_zone'] == z).sum()
    print(f"  Zone {z:2d}  (n={n_in_zone:3d} events):  ε = {e:.4f} days")

print(f"\nAdaptive ε results:  AF = {n_af_adap},  BG = {n_bg_adap}")
print(f"Fixed  ε results:    AF = {N_AF},  BG = {N_BG}")

# Agreement between fixed and adaptive methods
agree = (df['event_type'] == df_adaptive['event_type_adaptive']).sum()
print(f"\nAgreement between fixed-ε and adaptive-ε: "
      f"{agree}/{len(df)} = {100*agree/len(df):.1f}%")

# COV for adaptive BG
cov_adap_bg, _ = coefficient_of_variation(
    df_adaptive[df_adaptive['event_type_adaptive'] == 'BG']['time']
)
print(f"COV of adaptive-ε BG: {cov_adap_bg:.4f}  (target ≈ 1.0)")

---
## Cell 14: Export Declustered Catalog and Final Summary

In [ ]:
# =============================================================================
# CELL 14 — Export and Final Summary
# =============================================================================

# ── Export catalogs ───────────────────────────────────────────────────────────
export_cols = ['event', 'time', 'latitude', 'longitude', 'magnitude', 'depth',
               'som_zone', 'cluster_global', 'event_type']

df[export_cols].to_csv('outputs/NZ_catalog_labeled.csv', index=False)
print("✓  Full labeled catalog      → outputs/NZ_catalog_labeled.csv")

df[df['event_type'] == 'BG'][export_cols].to_csv(
    'outputs/NZ_background.csv', index=False)
print(f"✓  Background catalog (n={N_BG}) → outputs/NZ_background.csv")

df[df['event_type'] == 'AF'][export_cols].to_csv(
    'outputs/NZ_aftershocks.csv', index=False)
print(f"✓  Aftershock catalog (n={N_AF}) → outputs/NZ_aftershocks.csv")

# ── Parameter record ─────────────────────────────────────────────────────────
run_params = pd.Series({
    'Mc'               : Mc,
    'GR_b_value'       : round(b_value, 4),
    'GR_a_value'       : round(a_value, 4),
    'SOM_grid'         : f'{SOM_GRID_X}x{SOM_GRID_Y}',
    'SOM_iterations'   : SOM_ITER,
    'SOM_sigma0'       : SOM_SIGMA,
    'SOM_lr0'          : SOM_LR,
    'SOM_final_QE'     : round(final_qe, 6),
    'T_DBSCAN_eps_days': EPS_DAYS,
    'T_DBSCAN_min_pts' : MIN_PTS,
    'N_total'          : N_TOTAL,
    'N_AF'             : int(N_AF),
    'N_BG'             : int(N_BG),
    'N_clusters'       : N_CLUSTERS,
    'COV_total'        : round(cov_T, 4),
    'COV_AF'           : round(cov_A, 4) if not np.isnan(cov_A) else 'nan',
    'COV_BG'           : round(cov_B, 4) if not np.isnan(cov_B) else 'nan',
})
run_params.to_csv('outputs/run_parameters.csv', header=['value'])
print("✓  Run parameters            → outputs/run_parameters.csv")

# ── Final printed summary ─────────────────────────────────────────────────────
print()
print("═" * 60)
print("  FINAL DECLUSTERING SUMMARY")
print("  New Zealand Earthquake Catalog — SOM–DBSCAN Method")
print("═" * 60)
print(f"""
  PREPROCESSING
    Magnitude of Completeness Mc : {Mc:.2f}
    G-R b-value                  : {b_value:.3f}
    Events retained              : {len(df)}  of {len(df_raw)}

  STAGE 1 — SOM (Spatial Zoning)
    Grid size                    : {SOM_GRID_X}×{SOM_GRID_Y}  ({SOM_GRID_X*SOM_GRID_Y} neurons)
    Training iterations          : {SOM_ITER}
    Initial learning rate α₀     : {SOM_LR}
    Initial neighbourhood σ₀     : {SOM_SIGMA:.1f}
    Final Quantization Error     : {final_qe:.6f}
    Active seismic zones         : {n_active_zones}

  STAGE 2 — T-DBSCAN (Temporal Clustering)
    Temporal radius ε            : {EPS_DAYS} day(s)
    Minimum points MinPts        : {MIN_PTS}

  RESULTS
    Total events                 : {N_TOTAL}
    Aftershock clusters (AF)     : {N_AF}  ({100*N_AF/N_TOTAL:.1f}%)
    Background events  (BG)      : {N_BG}  ({100*N_BG/N_TOTAL:.1f}%)
    Clusters identified          : {N_CLUSTERS}

  VALIDATION — COV
    COV Total                    : {cov_T:.4f}
    COV Aftershocks (target > 1) : {cov_A:.4f}
    COV Background  (target ≈ 1) : {cov_B:.4f}
    Hierarchy COV_A > COV_B      : {'YES ✓' if not np.isnan(cov_A) and not np.isnan(cov_B) and cov_A > cov_B else 'NO ✗  — adjust parameters'}
""")
print("═" * 60)
print("  All outputs written to outputs/ and figures/ directories.")
print("═" * 60)